In [22]:
from pathlib import Path
import mlflow
from mlflow import MlflowClient

In [23]:
import os 
import sys

if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append(Path.cwd())


In [24]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()
MODEL_NAME = "turbofan-rul"

In [25]:
runs = mlflow.search_runs(
    experiment_names=["rul-fd001"],
    order_by=["metrics.val_rmse ASC"],
)

runs = runs.dropna(subset=["tags.git_commit"])

runs[["run_id", "metrics.val_rmse", "metrics.test_rmse", "params.max_depth", "params.min_samples_leaf"]].head()

,run_id,metrics.val_rmse,metrics.test_rmse,params.max_depth,params.min_samples_leaf
0,7ac40b18aa084306a1d54a3e73d868c0,16.47,20.36,8,1
1,71db3f044a6d461a9d24750df4159638,16.47,20.36,8,1
3,38203a7d85584e578476b71cb17f117b,16.61,20.54,12,1
4,2008d7f1df514123ad79d290132451fc,16.61,20.54,12,1
10,a62121930d6f4f298491714a56aecde9,18.83,21.87,3,1


In [26]:
best_run_id = runs.iloc[0]["run_id"]

mv = mlflow.register_model(f"runs:/{best_run_id}/model", MODEL_NAME)
print(mv.name, mv.version, mv.run_id)

Successfully registered model 'turbofan-rul'.
2026/10/25 23:57:29 WARNING mlflow.tracking._model_registry.fluent: Run with id 7ac40b18aa084306a1d54a3e73d868c0 has no artifacts at artifact path 'model', registering model based on models:/m-1a723e2adcef4984ac9acaf8c067eef8 instead


turbofan-rul 1 7ac40b18aa084306a1d54a3e73d868c0


Created version '1' of model 'turbofan-rul'.


In [27]:
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)

Loading the model and proving its the right one

In [28]:
from sklearn.metrics import root_mean_squared_error
from turbofan.features import get_feature_columns
from turbofan.train import build_test_data, load_params

In [29]:
champ_mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

params_file = mlflow.artifacts.download_artifacts(
    run_id=champ_mv.run_id, artifact_path="params.yaml"
)

champ_params = load_params(Path(params_file))
test = build_test_data(champ_params)

preds = champion.predict(test[get_feature_columns()])
rmse = root_mean_squared_error(test["rul"], preds)
logged = client.get_run(champ_mv.run_id).data.metrics["test_rmse"]

print(f"recomputed: {rmse:.2f} logged in run: {logged}")

recomputed: 20.36 logged in run: 20.36


Tracing the model back to its origins

In [30]:
run = client.get_run(champ_mv.run_id)

print("champion version:", champ_mv.version)
print("from run: ", champ_mv.run_id)
print("git commit: ", run.data.tags["git_commit"])
print("data md5: ", run.data.tags["data_md5"])
print("settings: ", run.data.params)

champion version: 1
from run:  7ac40b18aa084306a1d54a3e73d868c0
git commit:  7ab1d08b6b86481d53913237f0ad7e80b19df2b0
data md5:  d837c3a078121f105697ffd3a1dff232.dir
settings:  {'window': '10', 'rul_cap': '125', 'n_estimators': '100', 'max_depth': '8', 'min_samples_leaf': '1', 'random_state': '42', 'n_features': '45'}


In [31]:
client.set_model_version_tag(MODEL_NAME, champ_mv.version, "promotion", "approved")

Registering a second candidate

In [32]:
second_run_id = runs.iloc[1]["run_id"]

mv2 = mlflow.register_model(f"runs:/{second_run_id}/model", MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, "challenger", mv2.version)

Registered model 'turbofan-rul' already exists. Creating a new version of this model...
2026/10/25 23:57:43 WARNING mlflow.tracking._model_registry.fluent: Run with id 71db3f044a6d461a9d24750df4159638 has no artifacts at artifact path 'model', registering model based on models:/m-e64007ede1d144fa8f06d75221c1dd43 instead


Created version '2' of model 'turbofan-rul'.


Promoting the challenger and rollback

In [33]:
# promote the challenger
client.set_registered_model_alias(MODEL_NAME, "champion", mv2.version)
print("champion is now v", client.get_model_version_by_alias(MODEL_NAME, "champion").version)

# Oops, roll back
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print("Champion is now v", client.get_model_version_by_alias(MODEL_NAME, "champion").version)


champion is now v 2
Champion is now v 1


In [34]:
client.delete_registered_model_alias(MODEL_NAME, "challenger")

In [35]:
print("registered models:", [m.name for m in client.search_registered_models()])

runs = mlflow.search_runs(
    experiment_names=["rul-fd001"],
    order_by=["start_time ASC"],
)
runs = runs.dropna(subset=["tags.git_commit"])
runs = runs[runs["status"] == "FINISHED"]

runs[["run_id", "start_time", "metrics.val_rmse", "params.max_depth"]]

registered models: ['turbofan-rul']


,run_id,start_time,metrics.val_rmse,params.max_depth
16,38203a7d85584e578476b71cb17f117b,2026-10-25 09:31:21.070000+00:00,16.61,12
17,71db3f044a6d461a9d24750df4159638,2026-10-25 09:36:08.805000+00:00,16.47,8
18,7ac40b18aa084306a1d54a3e73d868c0,2026-10-26 06:51:45.331000+00:00,16.47,8
19,a62121930d6f4f298491714a56aecde9,2026-10-26 06:52:15.098000+00:00,18.83,3
